# Subword Tokenizer

| **Tokenizer 방식** | **토큰 단위**                      | **vocab size** | **미등록 단어에 대한 가정**                                                                                  |
|---------------------|------------------------------------|----------------|-------------------------------------------------------------------------------------------------------------|
| **사전 기반**       | 알려진 단어/형태소의 결합           | unlimited       | - 알려진 단어/형태소의 결합이라고 가정<br>- 필요한 형태소 분석 가능<br>- 사전에 등록되지 않은 단어는 UNK 처리 |
| **sub-word**        | 알려진 글자 및 sub-word            | fixed           | - 알려진 sub-words로 분해<br>- 예: appear → app + ear<br>- 자주 등장하는 단어를 제대로 인식 가능<br>- UNK의 개수 최소화 |

In [4]:
import urllib.request
import os

def get_file(filename, origin):
    cache_dir = os.path.expanduser('~\\.torch\\dataset')
    os.makedirs(cache_dir, exist_ok=True)
    filepath = os.path.join(cache_dir, filename)

    if not os.path.exists(filepath):
        print(f'{origin} 파일 다운 중')
        urllib.request.urlretrieve(origin, filepath)

    return filepath

In [5]:
rating_train_path = get_file(
    'rating_train.txt',
    'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
)

rating_test_path = get_file(
    'rating_test.txt',
    'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt'
)

rating_train_path, rating_test_path

('C:\\Users\\playdata2\\.torch\\dataset\\rating_train.txt',
 'C:\\Users\\playdata2\\.torch\\dataset\\rating_test.txt')

In [6]:
import pandas as pd

ratings_train_df = pd.read_csv(rating_train_path, sep='\t')
ratings_test_df = pd.read_csv(rating_test_path, sep='\t')

display(ratings_train_df)
display(ratings_test_df)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0
...,...,...,...
49995,4608761,오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49996,5308387,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49997,9072549,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,5802125,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0


In [7]:
ratings_train_df = ratings_train_df.dropna(how='any')
ratings_test_df = ratings_test_df.dropna(how='any')

ratings_train_df.shape, ratings_test_df.shape

((149995, 3), (49997, 3))

In [8]:
with open('naver_review.txt', 'w', encoding='utf-8') as f:
    for doc in ratings_train_df['document'].values:
        f.write(doc + '\n')

In [11]:
import sentencepiece as spt

input = 'naver_review.txt'
vocab_size = 10000
model_prefix = 'naver_review'

cmd = f'--input={input} --model_prefix={model_prefix} --vocab_size={vocab_size}'
spt.SentencePieceTrainer.Train(cmd)

True

In [12]:
sp = spt.SentencePieceProcessor()
sp.Load(f'{model_prefix}.model')

for doc in ratings_train_df['document'].values[:3]:
    print(doc)
    print(sp.encode_as_pieces(doc))
    print(sp.encode_as_ids(doc))
    print()

아 더빙.. 진짜 짜증나네요 목소리
['▁아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']
[63, 877, 5, 31, 2024, 69, 1714]

흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
['▁흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오', '버', '연기', '조차', '▁가볍지', '▁않', '구나']
[1633, 8, 4932, 157, 1281, 33, 269, 62, 171, 577, 419, 1231, 7426, 756, 448]

너무재밓었다그래서보는것을추천한다
['▁너무', '재', '밓', '었다', '그래서', '보는', '것을', '추천', '한다']
[22, 380, 9759, 427, 3788, 519, 2540, 1958, 331]



In [13]:
print(sp.get_piece_size())
print(sp.GetPieceSize())

10000
10000


In [16]:
text = ratings_test_df['document'][100]
tokens = sp.encode_as_pieces(text)
id_tokens = sp.encode_as_ids(text)

print(text)
print(tokens)
print(id_tokens)

print(''.join(tokens).replace("▁", " ").strip())

print(sp.decode_pieces(tokens))
print(sp.decode_ids(id_tokens))

걸작은 몇안되고 졸작들만 넘쳐난다.
['▁걸작', '은', '▁몇', '안되고', '▁졸작', '들만', '▁넘', '쳐', '난다', '.']
[1069, 16, 651, 6510, 726, 3182, 164, 683, 1009, 4]
걸작은 몇안되고 졸작들만 넘쳐난다.
걸작은 몇안되고 졸작들만 넘쳐난다.
걸작은 몇안되고 졸작들만 넘쳐난다.


In [18]:
from tokenizers import BertWordPieceTokenizer

tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
vocab_size = 10000

tokenizer.train(
    files = ['naver_review.txt'],
    vocab_size = vocab_size,
    min_frequency = 5,
    show_progress = True
)

In [20]:
tokenizer.get_vocab()

{'##였습니다': 3541,
 '나오고': 3858,
 '효과': 5712,
 '##웃': 1656,
 '##겠는데': 7215,
 '않': 626,
 '##수를': 2964,
 '##졌음': 6492,
 '않아요': 8356,
 '허접': 2518,
 '##민의': 7066,
 '##생활': 5597,
 '자랑': 6196,
 '혼란': 7298,
 '상상력이': 9554,
 '영화감독': 8278,
 '박수': 6395,
 '어설프': 3438,
 '무서움': 7960,
 '최근에': 5499,
 '자세': 6100,
 '##칸': 1613,
 '진지하게': 7095,
 '있었': 2607,
 '렬': 382,
 '##쉬': 1721,
 '보다가': 2381,
 '##웨이': 6873,
 '했': 956,
 '개그맨': 7176,
 '지루하지': 4196,
 '##지않다': 7494,
 '##더빙': 6420,
 '돋': 297,
 '모습': 2335,
 '##무시': 6823,
 '##다운': 4233,
 '##짱': 1440,
 'SF': 3352,
 '최고의영화': 6160,
 '큐': 867,
 '##답답': 5221,
 '##담': 1236,
 '##라지': 9839,
 '##명의': 4236,
 '분위기는': 8299,
 '며': 440,
 '못하고': 3659,
 '재미를': 4035,
 '점에서': 8637,
 '##그렇게': 9496,
 '없나': 4303,
 '무색': 9444,
 '##빛': 1327,
 '영화보는': 4355,
 '떠나서': 4556,
 '다루': 5265,
 '커플': 5052,
 '큰': 869,
 '##가면': 7656,
 '적절한': 6806,
 '##겠네요': 4626,
 '##스케': 6666,
 '##1': 1425,
 '##준것': 4901,
 '이쁨': 7290,
 '있나요': 9819,
 '허나': 7650,
 '저': 737,
 '##했지만': 2777,
 '##칙': 1851,
 '##시키지': 

In [21]:
tokenizer.save_model('./', 'bert_word_piece_from_naver_review')

['./bert_word_piece_from_naver_review-vocab.txt']

In [22]:
text = ratings_test_df['document'][100]
encoded = tokenizer.encode(text)

print(encoded.tokens)
print(encoded.ids)

print(text)
print(tokenizer.decode(encoded.ids))

['걸작', '##은', '몇', '##안되고', '졸작', '##들만', '넘쳐', '##난다', '.']
[2759, 1101, 444, 9509, 2589, 3798, 8336, 2430, 16]
걸작은 몇안되고 졸작들만 넘쳐난다.
걸작은 몇안되고 졸작들만 넘쳐난다.


- 여태까지 진행한 사항은 **“네이버 영화리뷰 텍스트를 모델이 먹을 수 있는 숫자 시퀀스로 바꾸는 ‘토크나이저(서브워드 사전)’를 직접 만들고, 인코딩/디코딩이 되는지 확인”**한 것임.

크게 목적은 3가지이다.

1. **단어사전(vocab) 만들기**
    - 리뷰 전체를 보고 자주 나오는 글자/부분단어(서브워드)를 모아 **토큰 사전**을 만든다.
    - OOV(처음 보는 단어) 문제를 줄이기 위해 “서브워드” 단위로 쪼개는 방식을 사용.
2. **텍스트 → 숫자(IDs)로 변환**
    - `encode_as_ids()`(SentencePiece) / `tokenizer.encode().ids`(WordPiece)로 문장을 **정수 ID 시퀀스**로 바꿔서, 이후 **Embedding/RNN/Transformer** 같은 모델 입력으로 넣을 수 있게 함.
3. **디코딩으로 검증(정상 동작 확인)**
    - `decode_ids()` / `decode()`로 다시 문장으로 복원해 보면서 “토큰화가 제대로 되는지”, “공백/특수문자 처리 문제가 없는지”를 확인.

추가로, 이번에 **SentencePiece vs WordPiece**를 둘 다 해본 건:

- 같은 한국어 데이터에서도 토큰화 방식이 어떻게 달라지는지 비교하고,
- 나중에 BERT류(WordPiece)나 일반 서브워드 모델(SentencePiece)에 맞게 선택하려는 목적이다.

- **표준형 딥러닝 텍스트 파이프라인**
    - **텍스트**: 원문 문장/문서(모델이 직접 처리 못 하는 문자열)
    - **정규화/전처리**: 노이즈 감소(소문자화, 특수문자 처리, 공백 정리 등)로 입력 형태를 일관되게 만듦
    - **토큰화(SentencePiece/WordPiece 등)**: 문장을 단어/서브워드 단위로 쪼개 OOV를 줄이고 모델이 다룰 “토큰”을 만듦
    - **ID 시퀀스**: 토큰을 정수로 매핑해 모델 입력 가능한 숫자 시퀀스로 변환(= vocab 기반 인덱싱)
    - **padding/truncation**: 배치 학습을 위해 길이를 고정(maxlen)하고, 짧으면 채우고 길면 잘라냄
    - **(Embedding)**: 정수 ID를 저차원 실수 벡터로 변환해 의미/유사도 학습이 가능하게 함(dense representation)
    - **Encoder(RNN/CNN/Transformer)**: 시퀀스에서 문맥/패턴을 추출해 문장 표현(특징 벡터)을 생성
    - **출력층**: 목적에 맞게 예측(분류 확률, 회귀값, 다음 토큰 등)을 계산하는 마지막 레이어